# 02 — General Overview: The Respiratory Failure Crisis in Numbers

High-level EDA for J96 (respiratory failure) admissions in SUS-SP. No deep analysis — just the landscape.

**Research Question:** What is the scale and trajectory of the mortality crisis?

**Depends on:** `resp_failure_sih.parquet`, `hospital_tags.parquet`, `related_conditions.parquet`

In [1]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from shared import (
    load_resp_failure, load_hospital_tags, load_related_conditions,
    setup_plot_style, save_plot, save_metrics,
    COLORS, ERA_COLORS, ERA_LABELS, ERA_ORDER,
    RELATED_ICDS,
)

setup_plot_style()

resp = load_resp_failure()
tags = load_hospital_tags()
related = load_related_conditions()

# Fix ICU marker: MARCA_UTI values include empty/NaN that shouldn't count as ICU
resp["icu_used"] = resp["MARCA_UTI"].astype(str).str.strip().isin(["1", "2", "3", "4", "5", "6"]).astype(int)

# Filter to 2016+ (core analysis period)
resp = resp[resp["year"] >= 2016].copy()

print(f"J96 admissions (2016+): {len(resp):,}")
print(f"Deaths: {resp['MORTE'].sum():,} ({resp['MORTE'].mean():.1%})")
print(f"ICU usage: {resp['icu_used'].mean():.1%}")

J96 admissions (2016+): 116,374
Deaths: 38,384 (33.0%)
ICU usage: 0.0%


## 1. Scale — The Big Numbers

In [2]:
total_admissions = len(resp)
total_deaths = resp["MORTE"].sum()
total_bed_days = resp["DIAS_PERM"].sum()
total_icu_days = resp["icu_days"].sum()
total_cost = resp["VAL_TOT"].sum()
mean_los = resp["DIAS_PERM"].mean()
median_los = resp["DIAS_PERM"].median()
mortality_rate = resp["MORTE"].mean()
n_hospitals = resp["CNES"].nunique()
n_municipalities = resp["MUNIC_MOV"].nunique()

print("=" * 50)
print("RESPIRATORY FAILURE (J96) — SUS SÃO PAULO")
print("Period: 2016–2025")
print("=" * 50)
print(f"Total admissions:    {total_admissions:>10,}")
print(f"Total deaths:        {total_deaths:>10,}")
print(f"Mortality rate:      {mortality_rate:>10.1%}")
print(f"Total bed-days:      {total_bed_days:>10,}")
print(f"Total ICU days:      {total_icu_days:>10,}")
print(f"Mean LOS:            {mean_los:>10.1f} days")
print(f"Median LOS:          {median_los:>10.0f} days")
print(f"Total cost:          R$ {total_cost:>10,.0f}")
print(f"Hospitals:           {n_hospitals:>10,}")
print(f"Municipalities:      {n_municipalities:>10,}")

RESPIRATORY FAILURE (J96) — SUS SÃO PAULO
Period: 2016–2025
Total admissions:       116,374
Total deaths:            38,384
Mortality rate:           33.0%
Total bed-days:       1,125,335
Total ICU days:         407,892
Mean LOS:                   9.7 days
Median LOS:                   5 days
Total cost:          R$ 383,471,537
Hospitals:                  562
Municipalities:             321


## 2. Mortality Trend — The Central Chart

In [3]:
yearly = resp.groupby("year").agg(
    admissions=("MORTE", "count"),
    deaths=("MORTE", "sum"),
    mortality_rate=("MORTE", "mean"),
    mean_los=("DIAS_PERM", "mean"),
    total_cost=("VAL_TOT", "sum"),
    icu_rate=("icu_used", "mean"),
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 7))

# Volume bars
bars = ax1.bar(yearly["year"], yearly["admissions"], alpha=0.3, color=COLORS["volume"], label="Admissions")
ax1.set_xlabel("Year", fontsize=12)
ax1.set_ylabel("Admissions", color=COLORS["volume"], fontsize=12)
ax1.tick_params(axis="y", labelcolor=COLORS["volume"])

# Mortality line
ax2 = ax1.twinx()
ax2.plot(yearly["year"], yearly["mortality_rate"] * 100, color=COLORS["danger"],
         linewidth=3, marker="o", markersize=8, label="Mortality Rate", zorder=5)
ax2.set_ylabel("Mortality Rate (%)", color=COLORS["danger"], fontsize=12)
ax2.tick_params(axis="y", labelcolor=COLORS["danger"])

# COVID annotation
ax1.axvspan(2020, 2021.5, alpha=0.08, color="red", label="COVID period")
ax1.annotate("COVID", xy=(2020.5, ax1.get_ylim()[1] * 0.95), fontsize=10, color="red", alpha=0.6, ha="center")

# Annotate first and last mortality rates
first_yr = yearly.iloc[0]
last_yr = yearly.iloc[-1]
ax2.annotate(f"{first_yr['mortality_rate']:.1%}", xy=(first_yr["year"], first_yr["mortality_rate"] * 100),
             textcoords="offset points", xytext=(-30, 10), fontsize=11, fontweight="bold", color=COLORS["danger"])
ax2.annotate(f"{last_yr['mortality_rate']:.1%}", xy=(last_yr["year"], last_yr["mortality_rate"] * 100),
             textcoords="offset points", xytext=(10, 10), fontsize=11, fontweight="bold", color=COLORS["danger"])

fig.suptitle("Respiratory Failure (J96) — Admissions & Mortality Trend\nSUS São Paulo, 2016–2025",
             fontsize=14, fontweight="bold")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
ax1.set_xticks(yearly["year"])

save_plot(fig, "mortality_trend", prefix="02")

print("\nYearly mortality rates:")
for _, row in yearly.iterrows():
    print(f"  {int(row['year'])}: {row['mortality_rate']:.1%} ({int(row['admissions']):,} admissions, {int(row['deaths']):,} deaths)")

  Saved: 02_mortality_trend.png

Yearly mortality rates:
  2016: 31.5% (10,891 admissions, 3,428 deaths)
  2017: 31.0% (10,764 admissions, 3,338 deaths)
  2018: 30.0% (10,772 admissions, 3,236 deaths)
  2019: 31.4% (10,466 admissions, 3,290 deaths)
  2020: 33.7% (14,681 admissions, 4,944 deaths)
  2021: 31.3% (13,195 admissions, 4,135 deaths)
  2022: 34.4% (10,968 admissions, 3,771 deaths)
  2023: 33.3% (11,738 admissions, 3,908 deaths)
  2024: 37.2% (11,612 admissions, 4,321 deaths)
  2025: 35.6% (11,287 admissions, 4,013 deaths)


## 3. Demographics — Who Gets Respiratory Failure?

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution
resp["age_group"] = pd.cut(resp["age"], bins=[0, 1, 18, 40, 60, 75, 120],
                            labels=["<1", "1-17", "18-39", "40-59", "60-74", "75+"])
age_stats = resp.groupby("age_group", observed=True).agg(
    count=("MORTE", "count"),
    mortality=("MORTE", "mean"),
).reset_index()

bar_colors = [COLORS["primary"]] * len(age_stats)
axes[0].bar(age_stats["age_group"].astype(str), age_stats["count"], color=bar_colors, alpha=0.8)
ax_mort = axes[0].twinx()
ax_mort.plot(age_stats["age_group"].astype(str), age_stats["mortality"] * 100,
             color=COLORS["danger"], marker="D", linewidth=2, markersize=6)
ax_mort.set_ylabel("Mortality %", color=COLORS["danger"])
axes[0].set_title("Age Distribution & Mortality")
axes[0].set_xlabel("Age Group")
axes[0].set_ylabel("Admissions")

# Sex distribution
sex_labels = {"1": "Male", "3": "Female"}
resp["sex_label"] = resp["SEXO"].astype(str).map(sex_labels).fillna("Other")
sex_stats = resp.groupby("sex_label").agg(
    count=("MORTE", "count"),
    mortality=("MORTE", "mean"),
).reset_index()
sex_colors = [COLORS["primary"], COLORS["secondary"]] + [COLORS["muted"]]
axes[1].bar(sex_stats["sex_label"], sex_stats["count"], color=sex_colors[:len(sex_stats)], alpha=0.8)
for i, row in sex_stats.iterrows():
    axes[1].annotate(f"Mort: {row['mortality']:.1%}", xy=(i, row["count"]),
                     textcoords="offset points", xytext=(0, 5), ha="center", fontsize=9, color=COLORS["danger"])
axes[1].set_title("Sex Distribution")
axes[1].set_ylabel("Admissions")

# Admission type
adm_labels = {"01": "Elective", "02": "Emergency", "05": "Other"}
resp["adm_label"] = resp["CAR_INT"].astype(str).str.strip().map(adm_labels).fillna("Other")
adm_stats = resp.groupby("adm_label").agg(
    count=("MORTE", "count"),
    mortality=("MORTE", "mean"),
).reset_index().sort_values("count", ascending=False)
axes[2].bar(adm_stats["adm_label"], adm_stats["count"],
            color=[COLORS["warning"], COLORS["danger"], COLORS["muted"]][:len(adm_stats)], alpha=0.8)
for i, (_, row) in enumerate(adm_stats.iterrows()):
    axes[2].annotate(f"Mort: {row['mortality']:.1%}", xy=(i, row["count"]),
                     textcoords="offset points", xytext=(0, 5), ha="center", fontsize=9, color=COLORS["danger"])
axes[2].set_title("Admission Type")
axes[2].set_ylabel("Admissions")

fig.suptitle("J96 Demographics — SUS São Paulo 2016–2025", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "demographics", prefix="02")

print(f"Mean age: {resp['age'].mean():.1f} years")
print(f"Median age: {resp['age'].median():.0f} years")
print(f"Male: {(resp['SEXO'].astype(str) == '1').mean():.1%}")
print(f"Emergency: {(resp['CAR_INT'].astype(str).str.strip() == '02').mean():.1%}")

  Saved: 02_demographics.png
Mean age: 44.4 years
Median age: 53 years
Male: 53.3%
Emergency: 95.4%


## 4. J96 Subtypes — Acute vs Chronic vs Unspecified

In [5]:
subtype_labels = {
    "J960": "J96.0 Acute",
    "J961": "J96.1 Chronic",
    "J969": "J96.9 Unspecified",
}
resp["subtype_code"] = resp["DIAG_PRINC"].astype(str).str[:4]
resp["subtype_label"] = resp["subtype_code"].map(subtype_labels).fillna("Other")

subtype_yearly = resp.groupby(["year", "subtype_label"]).agg(
    count=("MORTE", "count"),
    mortality=("MORTE", "mean"),
).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Volume by subtype
for label, grp in subtype_yearly.groupby("subtype_label"):
    ax1.plot(grp["year"], grp["count"], marker="o", linewidth=2, label=label)
ax1.set_title("Admissions by J96 Subtype")
ax1.set_xlabel("Year")
ax1.set_ylabel("Admissions")
ax1.legend()

# Mortality by subtype
for label, grp in subtype_yearly.groupby("subtype_label"):
    ax2.plot(grp["year"], grp["mortality"] * 100, marker="o", linewidth=2, label=label)
ax2.set_title("Mortality Rate by J96 Subtype")
ax2.set_xlabel("Year")
ax2.set_ylabel("Mortality Rate (%)")
ax2.legend()

fig.suptitle("J96 Subtypes — Volume & Mortality Trends", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "subtypes", prefix="02")

print("\nSubtype distribution:")
for label, grp in resp.groupby("subtype_label"):
    print(f"  {label}: {len(grp):,} ({len(grp)/len(resp):.1%}), mortality {grp['MORTE'].mean():.1%}")

  Saved: 02_subtypes.png

Subtype distribution:
  J96.0 Acute: 98,919 (85.0%), mortality 33.6%
  J96.1 Chronic: 3,350 (2.9%), mortality 12.8%
  J96.9 Unspecified: 12,395 (10.7%), mortality 31.2%
  Other: 1,710 (1.5%), mortality 47.7%


## 5. Mortality by COVID Era

In [6]:
era_stats = resp[resp["covid_era"] != "unknown"].groupby("covid_era").agg(
    admissions=("MORTE", "count"),
    deaths=("MORTE", "sum"),
    mortality=("MORTE", "mean"),
    mean_los=("DIAS_PERM", "mean"),
    mean_age=("age", "mean"),
    emergency_rate=("is_emergency", "mean"),
    icu_rate=("icu_used", "mean"),
    mean_cost=("VAL_TOT", "mean"),
).reindex(ERA_ORDER)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
era_colors = [ERA_COLORS[e] for e in ERA_ORDER if e in era_stats.index]
era_labels_display = [ERA_LABELS[e] for e in ERA_ORDER if e in era_stats.index]

# Mortality
axes[0].bar(range(len(era_stats)), era_stats["mortality"] * 100, color=era_colors)
axes[0].set_xticks(range(len(era_stats)))
axes[0].set_xticklabels(era_labels_display, rotation=25, ha="right", fontsize=8)
axes[0].set_ylabel("Mortality Rate (%)")
axes[0].set_title("Mortality by Era")
for i, v in enumerate(era_stats["mortality"]):
    axes[0].text(i, v * 100 + 0.3, f"{v:.1%}", ha="center", fontweight="bold", fontsize=9)

# Volume
axes[1].bar(range(len(era_stats)), era_stats["admissions"], color=era_colors)
axes[1].set_xticks(range(len(era_stats)))
axes[1].set_xticklabels(era_labels_display, rotation=25, ha="right", fontsize=8)
axes[1].set_ylabel("Admissions")
axes[1].set_title("Volume by Era")

# Mean Age
axes[2].bar(range(len(era_stats)), era_stats["mean_age"], color=era_colors)
axes[2].set_xticks(range(len(era_stats)))
axes[2].set_xticklabels(era_labels_display, rotation=25, ha="right", fontsize=8)
axes[2].set_ylabel("Mean Age")
axes[2].set_title("Mean Age by Era")

# Mean LOS
axes[3].bar(range(len(era_stats)), era_stats["mean_los"], color=era_colors)
axes[3].set_xticks(range(len(era_stats)))
axes[3].set_xticklabels(era_labels_display, rotation=25, ha="right", fontsize=8)
axes[3].set_ylabel("Mean LOS (days)")
axes[3].set_title("Mean LOS by Era")

fig.suptitle("J96 Key Metrics by COVID Era", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "era_comparison", prefix="02")

print("\nEra comparison:")
print(era_stats.to_string())

  Saved: 02_era_comparison.png

Era comparison:
                  admissions  deaths  mortality   mean_los   mean_age  emergency_rate  icu_rate    mean_cost
covid_era                                                                                                   
pre_covid              44447   13842   0.311427  10.332148  41.430603        0.954305       0.0  3169.258319
covid_acute            26322    8529   0.324026   8.154927  50.856926        0.949738       0.0  2815.606661
post_covid_early       17248    5724   0.331865   9.826357  41.895582        0.961619       0.0  3636.220667
post_covid_late        28357   10289   0.362838   9.943330  44.486864        0.954403       0.0  3730.210541


## 6. Geography — Top Municipalities

In [7]:
city_stats = resp.groupby("MUNIC_MOV").agg(
    admissions=("MORTE", "count"),
    deaths=("MORTE", "sum"),
    mortality=("MORTE", "mean"),
    mean_los=("DIAS_PERM", "mean"),
    n_hospitals=("CNES", "nunique"),
).reset_index()

# Top 20 by volume
top_volume = city_stats.nlargest(20, "admissions")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Volume
ax1.barh(range(len(top_volume)), top_volume["admissions"].values, color=COLORS["primary"], alpha=0.8)
ax1.set_yticks(range(len(top_volume)))
ax1.set_yticklabels(top_volume["MUNIC_MOV"].values, fontsize=9)
ax1.set_xlabel("Admissions")
ax1.set_title("Top 20 Municipalities by Volume")
ax1.invert_yaxis()

# Mortality for high-volume municipalities (n>=50)
high_vol = city_stats[city_stats["admissions"] >= 50].nlargest(20, "mortality")
colors_mort = [COLORS["danger"] if m > mortality_rate else COLORS["muted"] for m in high_vol["mortality"]]
ax2.barh(range(len(high_vol)), high_vol["mortality"].values * 100, color=colors_mort, alpha=0.8)
ax2.set_yticks(range(len(high_vol)))
ax2.set_yticklabels(high_vol["MUNIC_MOV"].values, fontsize=9)
ax2.axvline(mortality_rate * 100, color="black", linestyle="--", alpha=0.5, label=f"State avg: {mortality_rate:.1%}")
ax2.set_xlabel("Mortality Rate (%)")
ax2.set_title("Highest Mortality (municipalities with ≥50 admissions)")
ax2.legend()
ax2.invert_yaxis()

fig.suptitle("J96 Geographic Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "geography", prefix="02")

print(f"\nTotal municipalities treating J96: {len(city_stats)}")
print(f"Top 5 municipalities account for {top_volume.head(5)['admissions'].sum() / total_admissions:.1%} of all admissions")

  Saved: 02_geography.png

Total municipalities treating J96: 321
Top 5 municipalities account for 43.1% of all admissions


## 7. Hospital Landscape

In [8]:
hosp_stats = resp.groupby("CNES").agg(
    admissions=("MORTE", "count"),
    mortality=("MORTE", "mean"),
    mean_los=("DIAS_PERM", "mean"),
).reset_index()
hosp_stats = hosp_stats.merge(tags, on="CNES", how="left")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Facility type
type_counts = hosp_stats["broad_type"].value_counts()
axes[0].barh(type_counts.index, type_counts.values, color=COLORS["primary"], alpha=0.8)
axes[0].set_title("Hospitals by Facility Type")
axes[0].set_xlabel("Number of Hospitals")

# ICU capacity level
icu_counts = hosp_stats["icu_capacity_level"].value_counts()
icu_colors = {"no_icu": COLORS["muted"], "small_icu": COLORS["warning"],
              "medium_icu": COLORS["primary"], "large_icu": COLORS["success"]}
bar_colors = [icu_colors.get(str(k), COLORS["muted"]) for k in icu_counts.index]
axes[1].barh(icu_counts.index.astype(str), icu_counts.values, color=bar_colors, alpha=0.8)
axes[1].set_title("Hospitals by ICU Capacity")
axes[1].set_xlabel("Number of Hospitals")

# Legal nature
nat_counts = hosp_stats["nat_jur_category"].value_counts()
axes[2].barh(nat_counts.index, nat_counts.values, color=COLORS["secondary"], alpha=0.8)
axes[2].set_title("Hospitals by Legal Nature")
axes[2].set_xlabel("Number of Hospitals")

fig.suptitle("J96 Hospital Landscape — 562 Hospitals", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "hospital_landscape", prefix="02")

# Mortality by hospital type
print("\nMortality by facility type:")
for ftype, grp in resp.merge(tags[["CNES", "broad_type"]], on="CNES", how="left").groupby("broad_type"):
    print(f"  {ftype}: {grp['MORTE'].mean():.1%} ({len(grp):,} admissions)")

print("\nMortality by ICU capacity:")
for icu_lvl, grp in resp.merge(tags[["CNES", "icu_capacity_level"]], on="CNES", how="left").groupby("icu_capacity_level"):
    print(f"  {icu_lvl}: {grp['MORTE'].mean():.1%} ({len(grp):,} admissions)")

  Saved: 02_hospital_landscape.png

Mortality by facility type:


  hospital_especializado: 25.0% (6,114 admissions)
  hospital_geral: 33.6% (104,559 admissions)
  other: 26.8% (4,569 admissions)
  pronto_socorro: 40.2% (1,028 admissions)
  unidade_mista: 46.2% (104 admissions)

Mortality by ICU capacity:
  no_icu: 40.6% (59,520 admissions)
  small_icu: 26.7% (36,118 admissions)
  medium_icu: 22.2% (17,752 admissions)
  large_icu: 21.5% (2,984 admissions)


/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_63455/424011062.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for icu_lvl, grp in resp.merge(tags[["CNES", "icu_capacity_level"]], on="CNES", how="left").groupby("icu_capacity_level"):


## 8. Related Conditions — J18, J80, J44 Alongside J96

In [9]:
related_filt = related[related["year"] >= 2016].copy()
related_filt["MORTE"] = pd.to_numeric(related_filt["MORTE"], errors="coerce").fillna(0).astype(int)

related_yearly = related_filt.groupby(["year", "icd3"]).agg(
    admissions=("MORTE", "count"),
    mortality=("MORTE", "mean"),
).reset_index()

j96_yearly = resp.groupby("year").agg(
    admissions=("MORTE", "count"),
    mortality=("MORTE", "mean"),
).reset_index()
j96_yearly["icd3"] = "J96"

all_resp = pd.concat([related_yearly, j96_yearly], ignore_index=True)

icd_labels = {"J18": "J18 Pneumonia", "J44": "J44 COPD", "J80": "J80 ARDS", "J96": "J96 Resp Failure"}
icd_colors = {"J18": COLORS["primary"], "J44": COLORS["warning"],
              "J80": COLORS["secondary"], "J96": COLORS["danger"]}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for icd3, grp in all_resp.groupby("icd3"):
    grp = grp.sort_values("year")
    label = icd_labels.get(icd3, icd3)
    color = icd_colors.get(icd3, COLORS["muted"])
    ax1.plot(grp["year"], grp["admissions"], marker="o", linewidth=2, label=label, color=color)
    ax2.plot(grp["year"], grp["mortality"] * 100, marker="o", linewidth=2, label=label, color=color)

ax1.set_title("Admissions Trend")
ax1.set_xlabel("Year")
ax1.set_ylabel("Admissions")
ax1.legend()

ax2.set_title("Mortality Rate Trend")
ax2.set_xlabel("Year")
ax2.set_ylabel("Mortality Rate (%)")
ax2.legend()

fig.suptitle("Respiratory Conditions — Volume & Mortality Trends (2016–2025)", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "related_conditions", prefix="02")

print("\n2024 vs 2016 comparison:")
for icd3 in ["J96", "J18", "J44", "J80"]:
    grp = all_resp[all_resp["icd3"] == icd3]
    y16 = grp[grp["year"] == 2016]
    y24 = grp[grp["year"] == 2024]
    if len(y16) > 0 and len(y24) > 0:
        vol_change = (y24["admissions"].values[0] - y16["admissions"].values[0]) / y16["admissions"].values[0]
        mort_change = y24["mortality"].values[0] - y16["mortality"].values[0]
        print(f"  {icd_labels.get(icd3, icd3)}: volume {vol_change:+.0%}, mortality {mort_change:+.1%}pp")

  Saved: 02_related_conditions.png

2024 vs 2016 comparison:
  J96 Resp Failure: volume +7%, mortality +5.7%pp
  J18 Pneumonia: volume -5%, mortality +1.5%pp
  J44 COPD: volume +10%, mortality +1.2%pp
  J80 ARDS: volume +45%, mortality +1.4%pp


## 9. LOS Distribution

In [10]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Overall LOS distribution (capped at 60 for visualization)
los_capped = resp["DIAS_PERM"].clip(upper=60)
ax1.hist(los_capped, bins=60, color=COLORS["primary"], alpha=0.7, edgecolor="white")
ax1.axvline(resp["DIAS_PERM"].median(), color=COLORS["danger"], linestyle="--",
            label=f"Median: {resp['DIAS_PERM'].median():.0f} days")
ax1.axvline(resp["DIAS_PERM"].mean(), color=COLORS["warning"], linestyle="--",
            label=f"Mean: {resp['DIAS_PERM'].mean():.1f} days")
ax1.set_title("Length of Stay Distribution")
ax1.set_xlabel("Days (capped at 60)")
ax1.set_ylabel("Admissions")
ax1.legend()

# LOS-mortality relationship
los_buckets = pd.cut(resp["DIAS_PERM"], bins=[0, 1, 3, 7, 14, 30, 999],
                     labels=["0-1d", "2-3d", "4-7d", "8-14d", "15-30d", ">30d"])
los_mort = resp.groupby(los_buckets, observed=True).agg(
    count=("MORTE", "count"),
    mortality=("MORTE", "mean"),
).reset_index()

ax2.bar(los_mort["DIAS_PERM"].astype(str), los_mort["count"], color=COLORS["primary"], alpha=0.4)
ax2_mort = ax2.twinx()
ax2_mort.plot(los_mort["DIAS_PERM"].astype(str), los_mort["mortality"] * 100,
              color=COLORS["danger"], marker="D", linewidth=2, markersize=8)
ax2_mort.set_ylabel("Mortality Rate (%)", color=COLORS["danger"])
ax2.set_title("LOS Buckets & Mortality")
ax2.set_xlabel("Length of Stay")
ax2.set_ylabel("Admissions")

fig.suptitle("J96 Length of Stay Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "los_distribution", prefix="02")

print("\nLOS statistics:")
print(f"  Mean: {resp['DIAS_PERM'].mean():.1f}, Median: {resp['DIAS_PERM'].median():.0f}")
print(f"  P90: {resp['DIAS_PERM'].quantile(0.9):.0f}, P95: {resp['DIAS_PERM'].quantile(0.95):.0f}")
print(f"  Stays > 30 days: {(resp['DIAS_PERM'] > 30).mean():.1%}")

  Saved: 02_los_distribution.png

LOS statistics:
  Mean: 9.7, Median: 5
  P90: 26, P95: 32
  Stays > 30 days: 6.0%


## 10. Comorbidity Landscape

In [11]:
comorbidity_cols = [c for c in resp.columns if c.startswith("comorbidity_") and c != "comorbidity_count"]

comorb_stats = []
for col in comorbidity_cols:
    name = col.replace("comorbidity_", "").replace("_", " ").title()
    prevalence = resp[col].mean()
    mort_with = resp[resp[col] == 1]["MORTE"].mean()
    mort_without = resp[resp[col] == 0]["MORTE"].mean()
    comorb_stats.append({"comorbidity": name, "prevalence": prevalence,
                         "mort_with": mort_with, "mort_without": mort_without,
                         "mort_diff": mort_with - mort_without})

comorb_df = pd.DataFrame(comorb_stats).sort_values("prevalence", ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Prevalence
ax1.barh(comorb_df["comorbidity"], comorb_df["prevalence"] * 100, color=COLORS["primary"], alpha=0.8)
ax1.set_xlabel("Prevalence (%)")
ax1.set_title("Comorbidity Prevalence in J96 Patients")

# Mortality impact
colors_impact = [COLORS["danger"] if d > 0 else COLORS["success"] for d in comorb_df["mort_diff"]]
ax2.barh(comorb_df["comorbidity"], comorb_df["mort_diff"] * 100, color=colors_impact, alpha=0.8)
ax2.axvline(0, color="black", linewidth=0.5)
ax2.set_xlabel("Mortality Difference (pp)")
ax2.set_title("Mortality Impact (with vs. without)")

fig.suptitle("J96 Comorbidity Landscape", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "comorbidities", prefix="02")

print("\nComorbidity mortality impact:")
for _, row in comorb_df.sort_values("mort_diff", ascending=False).iterrows():
    print(f"  {row['comorbidity']}: prevalence {row['prevalence']:.1%}, "
          f"mortality with {row['mort_with']:.1%} vs without {row['mort_without']:.1%} ({row['mort_diff']:+.1%}pp)")

  Saved: 02_comorbidities.png

Comorbidity mortality impact:
  Stroke: prevalence 0.9%, mortality with 70.8% vs without 32.6% (+38.1%pp)
  Sepsis: prevalence 2.4%, mortality with 68.7% vs without 32.1% (+36.6%pp)
  Kidney Failure: prevalence 2.7%, mortality with 64.2% vs without 32.1% (+32.1%pp)
  Pulmonary Fibrosis: prevalence 0.2%, mortality with 59.1% vs without 32.9% (+26.2%pp)
  Heart Failure: prevalence 1.7%, mortality with 56.4% vs without 32.6% (+23.8%pp)
  Diabetes: prevalence 2.6%, mortality with 55.1% vs without 32.4% (+22.7%pp)
  Ischemic Heart: prevalence 0.5%, mortality with 52.8% vs without 32.9% (+19.9%pp)
  Copd: prevalence 2.6%, mortality with 50.3% vs without 32.5% (+17.8%pp)
  Hypertension: prevalence 5.1%, mortality with 49.1% vs without 32.1% (+17.0%pp)
  Covid: prevalence 1.3%, mortality with 38.9% vs without 32.9% (+6.0%pp)
  Obesity: prevalence 0.5%, mortality with 34.2% vs without 33.0% (+1.2%pp)
  Asthma: prevalence 1.9%, mortality with 8.1% vs without 33.5% 

## 11. Summary Metrics

In [12]:
metrics = {
    "total_admissions": total_admissions,
    "total_deaths": int(total_deaths),
    "mortality_rate": round(mortality_rate, 4),
    "total_bed_days": int(total_bed_days),
    "total_icu_days": int(total_icu_days),
    "mean_los": round(mean_los, 1),
    "median_los": int(median_los),
    "total_cost_brl": round(total_cost, 2),
    "n_hospitals": n_hospitals,
    "n_municipalities": n_municipalities,
    "mean_age": round(resp["age"].mean(), 1),
    "pct_male": round((resp["SEXO"].astype(str) == "1").mean(), 4),
    "pct_emergency": round((resp["CAR_INT"].astype(str).str.strip() == "02").mean(), 4),
    "mortality_by_era": {
        era: round(era_stats.loc[era, "mortality"], 4)
        for era in ERA_ORDER if era in era_stats.index
    },
    "mortality_by_year": {
        str(int(r["year"])): round(r["mortality_rate"], 4)
        for _, r in yearly.iterrows()
    },
    "mortality_change_pp": round(
        yearly.iloc[-1]["mortality_rate"] - yearly.iloc[0]["mortality_rate"], 4
    ),
}

save_metrics(metrics, "02_general_overview")

print("\n" + "=" * 50)
print("KEY TAKEAWAYS")
print("=" * 50)
print(f"\n1. SCALE: {total_admissions:,} admissions, {int(total_deaths):,} deaths over 10 years")
print(f"2. MORTALITY CRISIS: {yearly.iloc[0]['mortality_rate']:.1%} → {yearly.iloc[-1]['mortality_rate']:.1%} "
      f"({metrics['mortality_change_pp']*100:+.1f}pp)")
print(f"3. COVID ERA: pre-COVID {era_stats.loc['pre_covid', 'mortality']:.1%} → "
      f"post-COVID late {era_stats.loc['post_covid_late', 'mortality']:.1%}")
print(f"4. HIGH-RISK: elderly patients, emergency admissions dominate")
print(f"5. COST: R$ {total_cost:,.0f} total ({resp['VAL_TOT'].mean():,.0f} per admission)")

  Saved metrics: 02_general_overview.json

KEY TAKEAWAYS

1. SCALE: 116,374 admissions, 38,384 deaths over 10 years
2. MORTALITY CRISIS: 31.5% → 35.6% (+4.1pp)
3. COVID ERA: pre-COVID 31.1% → post-COVID late 36.3%
4. HIGH-RISK: elderly patients, emergency admissions dominate
5. COST: R$ 383,471,537 total (3,295 per admission)
